In [0]:
import re
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import col
from delta.tables import DeltaTable

# Feature Engineering

In [0]:
%skip
SELECT * FROM fraud_detection_project.silver_layer.transaction_events LIMIT(100)

In [0]:
# Reading important tables
df_transaction = spark.read.table("fraud_detection_project.silver_layer.transaction_events") 
df_customer = spark.read.table("fraud_detection_project.silver_layer.customer_profiles")

# 
df_transaction.printSchema()
df_customer.printSchema()

In [0]:
# Selecting only important columns
df_customer_slim = df_customer.select(
    "customer_id",
    "card_number"
)

# Joining customer and transaction tables
df = df_transaction.join(
    df_customer_slim,
    df_customer_slim.card_number == df_transaction.card_id
)

hour_window = Window.partitionBy("card_id").orderBy(F.expr("try_cast(txn_timestamp as timestamp)").cast("long")).rangeBetween(-3600, 0)

day_window =   Window.partitionBy("card_id").orderBy(F.expr("try_cast(txn_timestamp as timestamp)").cast("long")).rangeBetween(-86400, 0)

week_window =  Window.partitionBy("card_id").orderBy(F.expr("try_cast(txn_timestamp as timestamp)").cast("long")).rangeBetween(-604800, 0)

month_window = Window.partitionBy("card_id").orderBy(F.expr("try_cast(txn_timestamp as timestamp)").cast("long")).rangeBetween(-2592000, 0)

# Creating Velocity Features
df = df.withColumn("count_txn_1h", F.count("transaction_id").over(hour_window))
df = df.withColumn("count_txn_24h", F.count("transaction_id").over(day_window))
df = df.withColumn("count_txn_7d", F.count("transaction_id").over(week_window))
df = df.withColumn("count_txn_30d", F.count("transaction_id").over(month_window))

# Creating Avarage Features 
# df = df.withColumn("avg_amount_24h", F.avg("count_txn"))


In [0]:
df.createOrReplaceTempView("df1")

A Lógica do meu código está quebrada. Não está retornando o esperado.

In [0]:
%sql
SELECT count_txn_1h, count_txn_24h, count_txn_7d, count_txn_30d FROM df1 DESC LIMIT(10) 